# Drift Monitoring — Inference Monitoring Notebook

> 📊 **Note**: This notebook contains interactive output cells (Evidently reports and Plotly charts). They render fully in JupyterLab; static notebook viewers may omit the JavaScript output.

This notebook provides end-to-end drift monitoring for the deployed **bank-marketing** endpoint. The workshop stack provisions the Athena tables, inference-capture path, and MLflow application. The notebook generates test traffic, computes drift, and writes monitoring results directly to Athena and MLflow; it creates no infrastructure.

You focus on the parts that need a data scientist:

1. **Setup** — load the workshop environment, verify the monitoring tables, and resolve the registered model baseline.
2. **Generate drifted data** *(dev/test only)* — create traffic that should trip the detector.
3. **Send predictions** — invoke the endpoint so there is traffic to monitor.
4. **Apply ground truth** — simulate labels for model-drift metrics.
5. **Detect drift** — compute data drift and model drift with Evidently, then write the results to Athena and MLflow.

## Prerequisites

This notebook assumes the following resources already exist. The workshop stack and earlier labs create them.

| Prerequisite | How it is created |
|---|---|
| SageMaker domain, S3 data bucket, MLflow application, `bank_marketing` Athena tables (`training_data`, `evaluation_data`, `inference_responses`, `monitoring_responses`, and `ground_truth_updates`), and inference-capture path | Workshop stack (`scripts/deploy-workshop.sh` and `templates/`) |
| A trained and **Approved** model in the `bank-prediction-XGBoostModel` model package group | Lab 3A / 3C |
| A live `bank-marketing-<timestamp>` endpoint that logs to `inference_responses` | Lab 3A |

The shared `.env` provides `DATA_S3_BUCKET` and `AWS_DEFAULT_REGION`. The notebook resolves the remaining values from the workshop stack outputs and discovers the most recent InService `bank-marketing-*` endpoint.

Running this notebook invokes the endpoint and writes ground-truth and monitoring rows. It does not create infrastructure or change QuickSight resources.

## 1. Setup

Load the shared workshop environment and initialize the monitoring session. The session resolves the endpoint, AWS clients, schema, and configured thresholds used by the remaining sections.

In [ ]:
from src.config.notebook import ensure_lab5_package, load_shared_environment

env_path = load_shared_environment()
package_path = ensure_lab5_package()
from src.drift_monitoring.monitoring_session import MonitoringSession

session = MonitoringSession()
print(f'Environment: {env_path}')
print(f'Package: {package_path}')
session.summary()

### 1.1 Monitoring tables (provisioned by the workshop stack)

The Athena tables this notebook reads and writes are created in the `bank_marketing` database by the workshop CloudFormation stack. The next cell checks the Glue catalog without changing it. If a table is missing, redeploy the workshop stack instead of creating an ad hoc table from this notebook.

In [ ]:
table_status = session.verify_monitoring_tables()
table_status

if not all(table_status.values()):
    raise RuntimeError(f'Missing workshop table(s): {[k for k, v in table_status.items() if not v]}')

### 1.2 What this notebook does

The notebook sends predictions to the live endpoint, applies simulated ground truth, computes **data drift** and **model drift** with Evidently, writes the results to `monitoring_responses`, and logs metrics and reports to **MLflow**. The monitoring tables then back the **QuickSight** governance dashboard in Lab 5E.

Before generating traffic, the next cell resolves the baseline lineage registered with the model serving the endpoint. Data drift uses the model's training snapshot. Model drift uses its held-out evaluation snapshot, re-scored with the same model artifact.

In [ ]:
lineage = session.load_baseline()
lineage

## 2. Generate Drifted Test Data *(dev/test only)*

Generate a drifted CSV that the detector should flag, so Section 5 has something to find. In production, you would skip this section because real traffic supplies the drift.

The generator samples rows from `training_data` and applies the shifts configured in `src/config/config.yaml`. The random seed and sample count are also configured there, so the generated test data is reproducible.

In [ ]:
drift_csv = session.generate_drifted_data()
print(f'Generated: {drift_csv}')

## 3. Send Predictions to the Endpoint

Send the drifted samples from Section 2 to the deployed endpoint. The endpoint's inference-capture path logs each prediction to `inference_responses` through SQS and Lambda.

The three short calls prepare the monitoring context, send 101 samples, and wait for the captured predictions before continuing.

In [ ]:
window = session.start_run()
print(f'Run: {window.monitoring_run_id}')
print(f'Lower bound (Athena UTC): {window.started_at}')

In [ ]:
traffic = session.send_drift_traffic(count=101, csv_path=drift_csv)
traffic

In [ ]:
capture = session.close_capture(timeout_seconds=240, poll_seconds=10)
capture

## 4. Apply Ground Truth

In production, labels arrive asynchronously from investigations, manual review, or delayed outcome observation. For dev/test, this section simulates labels so model-drift metrics have outcomes to score.

The simulator copies each prediction and flips a deterministic fraction of labels. The call applies those labels to the predictions sent in Section 3 and returns coverage, which must match the number of captured predictions before drift detection continues.

In [ ]:
ground_truth = session.apply_simulated_ground_truth(seed=42)
ground_truth

## 5. Detect Drift

This is the core of the notebook. It performs two independent checks, both traceable to the model and data snapshots registered in `baseline.json`:

- **Data drift** compares endpoint inputs with `training_data`.
- **Model drift** compares classification metrics from labeled endpoint predictions with the registered model's `evaluation_data`.

### 5.1 Data drift

This notebook overrides Evidently's automatic test selection. It pins Jensen-Shannon distance for numeric and categorical columns in every new report. It uses a threshold of `0.1`.

#### How to read drift scores

- **Jensen-Shannon distance (JSD):** `0` means the distributions are the same. A higher value means more drift.
- **Kolmogorov-Smirnov (KS):** This test compares numeric distributions. It returns a p-value. A lower p-value means stronger evidence of drift.
- **Normalized Wasserstein distance:** This measures how far a numeric distribution moves. A higher value means more drift.
- **Chi-square and Z-test:** Chi-square compares category frequencies. A Z-test compares binary proportions. Both return p-values. A lower p-value means stronger evidence of drift.

For the pinned Evidently 0.7.x family, automatic selection uses KS for numeric data with 1,000 or fewer observations. It uses normalized Wasserstein distance for numeric data with more than 1,000 observations. For categorical data with 1,000 or fewer observations, it uses Chi-square when there are more than two categories and Z-test for binary data. It uses Jensen-Shannon distance for categorical data with more than 1,000 observations. These are Evidently's general defaults. They are alternate behavior here because this notebook pins JSD instead.

Distances and p-values point in opposite directions. Higher distances indicate more drift. Lower p-values indicate more evidence of drift.

The charts therefore use normalized `drift_magnitude`. It accounts for each test's threshold and direction. `1.0` is the boundary. Values above `1.0` indicate drift, regardless of the raw-score direction. Sampling variation can move scores up or down between runs. Do not expect a strictly monotonic trend.

The dataset verdict uses a separate rule: `drifted_columns_share >= DATA_DRIFT_THRESHOLD` (default 20%). Only features with at least 50 non-null observations on both sides enter that denominator. All-null or undersampled features are persisted with null score, magnitude, and verdict rather than zero.

In [ ]:
import pandas as pd

data_drift = session.run_data_drift(baseline_limit=5000)
print({k: data_drift[k] for k in (
    'drift_detected', 'drifted_columns_count',
    'drifted_columns_share', 'features_analyzed', 'sample_size')})
pd.DataFrame(data_drift['per_column']).T.sort_values(
    'drift_magnitude', ascending=False, na_position='last')

### 5.2 Model drift

Model drift compares classification metrics. The baseline `(target, prediction)` pairs come from `evaluation_data`, re-scored with the registered model artifact. The current `(ground_truth, prediction)` pairs come from the labeled endpoint predictions produced in Sections 3 and 4.

Current labels, predicted labels, and positive-class probabilities are cleaned together to preserve row pairing. ROC-AUC requires both ground-truth classes, and Evidently's classification comparison also requires both predicted classes on each side.

If either requirement is unavailable, the cell prints the reason and leaves model fields null. Otherwise, the monitor reports model drift when relative ROC-AUC degradation reaches `MODEL_DRIFT_THRESHOLD` (default 5%).

In [ ]:
model_drift = session.run_model_drift(prediction_threshold=0.5)
{k: v for k, v in model_drift.items() if k != 'report'}

if model_drift['report']:
    display(model_drift['report']['snapshot'])
else:
    print(f"Model comparison skipped: {model_drift['skip_reason']}")

### 5.3 Log results to MLflow and Athena

The final call logs the data-drift and model-drift metrics and report artifacts to MLflow, then inserts the monitoring result into `monitoring_responses` for the Lab 5E QuickSight dashboard. Per-feature results retain the raw score, normalized magnitude, method, threshold, direction, sample counts, comparability, and reason when a score is unavailable.

In [ ]:
persisted = session.persist_results(log_to_mlflow=True)
persisted

## 6. Review Results in Lab 5E

Continue to Lab 5E to create the QuickSight governance dashboard backed by the workshop monitoring tables. The feature charts use `drift_magnitude`, the dataset charts use `drifted_columns_share`, and unavailable values remain null. Reference lines mark feature magnitude `1.0` and the configured dataset-share threshold.

## Teardown

This notebook creates no infrastructure. The generated CSV is local; inference, ground-truth, and monitoring rows belong to the workshop tables. Delete the endpoint from Lab 3A when all Lab 5 work is complete, then follow the workshop stack cleanup instructions.